## boilerplate

In [ ]:
import torch
import torchaudio
import torchaudio.functional as F
import numpy as np
import os
import gc
import re
from pathlib import Path
from natsort import natsorted
import librosa
import transformers
from sklearn.metrics import pairwise_distances
import warnings
print(f"Torch Version: {torch.__version__}")
# This should now be 2.6.0 or higher
print(f"Is CUDA available: {torch.cuda.is_available()}")
warnings.filterwarnings("ignore")



BATCH_SIZE = 16  # Can increase since 2s clips are small
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device is {DEVICE}")
DTYPE = torch.float32  # to ensure comaptibility across all models or 
BASE_DIR = Path("/home/amin/Research/Representation/Movie/data/Setareh/Data")
OUTPUT_PATH = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings")

In [2]:
# --- CONFIG ---
SEGMENT_DURATIONS = [1.0, 2.0, 5.0, 10.0]  
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
# ----------------

def extract_part_number(filename):
    match = re.search(r'part[_-]?(\d+)', filename.stem, re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None

def load_video(path, num_frames=16):
    from decord import VideoReader, cpu
    import decord
    decord.bridge.set_bridge("torch")
    vr = VideoReader(str(path), ctx=cpu(0))
    idx = np.linspace(0, len(vr) - 1, num_frames).astype(np.int64)
    frames = vr.get_batch(idx) 
    return frames.permute(0, 3, 1, 2) 

def load_audio(path, target_len, sr=16000):
    # 1. Load the audio natively (returns a PyTorch tensor, not numpy)
    wav, orig_sr = torchaudio.load(path)
    
    # 2. Convert stereo to mono by averaging across the channel dimension
    if wav.shape[0] > 1:
        wav = torch.mean(wav, dim=0, keepdim=True)
        
    # 3. Resample if the original sample rate doesn't match the target
    if orig_sr != sr:
        wav = F.resample(wav, orig_freq=orig_sr, new_freq=sr)
        
    # 4. Squeeze to 1D if your padding/trimming logic expects a flat array [Length]
    # (torchaudio loads as [Channels, Frames], so mono is [1, Length])
    wav = wav.squeeze(0)
    
    # Pad or trim to target length
    if wav.numel() < target_len:
        wav = torch.nn.functional.pad(wav, (0, target_len - wav.numel()))
    else:
        wav = wav[:target_len]
    
    return wav

def gather_pairs_for_segment(seg_sec):
    """Load video and audio pairs for a given segment duration"""
    all_videos = []
    all_audios = []
    
    for run_idx in range(1, 19):
        vid_chunk_dir = BASE_DIR / f"Video{run_idx}" / f"Video{run_idx}_chunks_{int(seg_sec)}s"
        aud_chunk_dir = BASE_DIR / f"Audio{run_idx}" / f"Audio{run_idx}_chunks_{int(seg_sec)}s"

        if not vid_chunk_dir.exists() or not aud_chunk_dir.exists():
            continue

        vids = natsorted(list(vid_chunk_dir.glob("*.mp4")))
        # Search for .wav (preferred) or fallback to .m4a if needed
        auds = natsorted(list(aud_chunk_dir.glob("*.wav")))
        if not auds: 
            auds = natsorted(list(aud_chunk_dir.glob("*.m4a")))

        if len(vids) == 0 or len(auds) == 0:
            continue

        vid_parts = {extract_part_number(v): v for v in vids if extract_part_number(v) is not None}
        aud_parts = {extract_part_number(a): a for a in auds if extract_part_number(a) is not None}

        common_parts = sorted(set(vid_parts.keys()) & set(aud_parts.keys()))

        for part_num in common_parts:
            all_videos.append(vid_parts[part_num])
            all_audios.append(aud_parts[part_num])

        if len(vids) != len(auds):
            print(f"Warning: Video {run_idx} has mismatched counts (video: {len(vids)}, audio: {len(auds)})")

    paired = list(zip(all_videos, all_audios))
    return paired, all_videos, all_audios

print(f"Helper functions loaded for segment-based extraction")


Helper functions loaded for segment-based extraction


## Videomaev2

In [ ]:

from transformers import AutoModel, AutoImageProcessor
import os

# Use the standard repo ID
MODEL_ID = "OpenGVLab/VideoMAEv2-Large"
# Point to the base cache directory, not the exact snapshot hash
CACHE_DIR = "/home/amin/hf_models/manual/"

try:
    # Adding _fast_init=False and device_map=None strictly forces standard RAM initialization
    model_v = AutoModel.from_pretrained(
        MODEL_ID, 
        cache_dir=CACHE_DIR,
        local_files_only=True, 
        trust_remote_code=True,
        low_cpu_mem_usage=False,
        _fast_init=False,        # <-- Disables meta tensor initialization
        device_map=None          # <-- Prevents accelerate from auto-mapping
    ).to(DTYPE).to(DEVICE).eval()
    
    processor_v = AutoImageProcessor.from_pretrained(
        MODEL_ID, 
        cache_dir=CACHE_DIR,
        local_files_only=True
    )
    print("VideoMAE Ready (Local)")
    
except Exception as e:
    print(f"Local load failed: {e}. Downloading from Hub...")
    # Apply the exact same safety overrides to the fallback
    model_v = AutoModel.from_pretrained(
        MODEL_ID, 
        trust_remote_code=True,
        low_cpu_mem_usage=False,
        _fast_init=False,
        device_map=None
    ).to(DTYPE).to(DEVICE).eval()
    
    processor_v = AutoImageProcessor.from_pretrained(MODEL_ID)
    print("VideoMAE Ready (Hub)")

# Loop through all segment durations
for seg_sec in SEGMENT_DURATIONS:
    print(f"\n{'='*60}")
    print(f"VideoMAE: Processing {int(seg_sec)}s segments")
    print(f"{'='*60}")
    
    # Gather data for this segment duration
    paired, all_videos, all_audios = gather_pairs_for_segment(seg_sec)
    
    if len(paired) == 0:
        print(f"No data found for {int(seg_sec)}s segments. Skipping.")
        continue
    
    print(f"Found {len(paired)} paired segments")
    
    # Create output directory
    out_dir = OUTPUT_PATH / "videomaev2-large" / f"{int(seg_sec)}s"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    vmae_embs = []
    with torch.inference_mode():
        for i, vp in enumerate(all_videos):
            frames = load_video(vp)
            inputs = processor_v(list(frames), return_tensors="pt")
            
            # Fix: Permute from [B, T, C, H, W] to [B, C, T, H, W]
            pixel_values = inputs['pixel_values']
            if pixel_values.ndim == 5 and pixel_values.shape[1] > pixel_values.shape[2]:
                # Detected [B, T, C, H, W], permute to [B, C, T, H, W]
                pixel_values = pixel_values.permute(0, 2, 1, 3, 4)
                inputs['pixel_values'] = pixel_values
            
            inputs = {k: v.to(DTYPE).to(DEVICE) for k, v in inputs.items()}
            out = model_v(**inputs)
            
            # Handle different return types
            if isinstance(out, torch.Tensor):
                # Direct tensor output - use as is or pool if needed
                if out.ndim == 3:  # [B, seq_len, hidden_dim]
                    h = out.mean(dim=1)  # Pool over sequence
                else:
                    h = out
            elif hasattr(out, 'last_hidden_state'):
                h = out.last_hidden_state.mean(dim=1)
            elif hasattr(out, 'pooler_output'):
                h = out.pooler_output
            else:
                # Fallback: assume first element is the hidden states
                h = out[0].mean(dim=1) if out[0].ndim == 3 else out[0]
            
            vmae_embs.append(h.cpu())
            
            if (i + 1) % 50 == 0:
                print(f"Processed {i + 1}/{len(all_videos)} segments...")

    videomae_out = torch.cat(vmae_embs).numpy()
    print(f"VideoMAE embeddings shape: {videomae_out.shape}")
    
    out_file = out_dir / "videomaev2-large_v.npy"
    np.save(out_file, videomae_out)
    print(f"Saved to: {out_file}\n")
    
    del vmae_embs, videomae_out
    gc.collect()

print("VideoMAE processing complete!")
del model_v, processor_v
gc.collect()
torch.cuda.empty_cache()


## WavLM Extraction

In [ ]:
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
import os

WAVLM_PATH = "/home/amin/hf_models/manual/models--microsoft--wavlm-large/snapshots/c1423ed94bb01d80a3f5ce5bc39f6026a0f4828c"

# Load model once (reuse for all segment durations)
try:
    model_a = WavLMModel.from_pretrained(WAVLM_PATH, local_files_only=True, use_safetensors=True).to(DTYPE).to(DEVICE).eval()
    processor_a = Wav2Vec2FeatureExtractor.from_pretrained(WAVLM_PATH, local_files_only=True)
    print("WavLM Model & Processor Ready")
except Exception as e:
    print(f"Error loading WavLM locally: {e}")
    print("Downloading from HuggingFace...")
    model_a = WavLMModel.from_pretrained("microsoft/wavlm-large").to(DTYPE).to(DEVICE).eval()
    processor_a = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-large")
    print("WavLM Model & Processor Ready")

# Loop through all segment durations
for seg_sec in SEGMENT_DURATIONS:
    print(f"\n{'='*60}")
    print(f"WavLM: Processing {int(seg_sec)}s segments")
    print(f"{'='*60}")
    
    # Gather data for this segment duration
    paired, all_videos, all_audios = gather_pairs_for_segment(seg_sec)
    
    if len(paired) == 0:
        print(f"No data found for {int(seg_sec)}s segments. Skipping.")
        continue
    
    print(f"Found {len(paired)} paired segments")
    
    # Create output directory
    out_dir = OUTPUT_PATH / "wavlm-large" / f"{int(seg_sec)}s"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    wavlm_embs = []
    target_len = int(seg_sec * 16000)

    with torch.inference_mode():
        for i, ap in enumerate(all_audios):
            wav = load_audio(ap, target_len)
            inp = processor_a(wav.cpu().numpy(), sampling_rate=16000, return_tensors="pt").to(DTYPE).to(DEVICE)
            out = model_a(**inp)
            # Mean pool over temporal dimension
            h = out.last_hidden_state.mean(dim=1)
            wavlm_embs.append(h.cpu())
            if (i + 1) % 50 == 0:
                print(f"Processed {i + 1}/{len(all_audios)} audio segments...")

    wavlm_out = torch.cat(wavlm_embs).numpy()
    print(f"WavLM embeddings shape: {wavlm_out.shape}")
    
    out_file = out_dir / "wavlm-large_a.npy"
    np.save(out_file, wavlm_out)
    print(f"Saved to: {out_file}\n")
    
    del wavlm_embs, wavlm_out
    gc.collect()

print("WavLM processing complete!")
del model_a, processor_a
gc.collect()
torch.cuda.empty_cache()


## AudioMAE-base Extraction

### **Methods Draft: AudioMAE Feature Extraction**

**Model & Architecture**

* **Model:** AudioMAE (Masked Autoencoders that Listen) [Huang et al., 2022].
* **Backbone:** ViT-Base (M params, ,  blocks).
* **Weights:** Pre-trained on AudioSet-2M [Gemmeke et al., 2017].
* **Checkpoint Choice:** **Self-Supervised** (not fine-tuned). We chose this to preserve rich acoustic variance and structural details (reconstruction objective) rather than collapsing features for classification, ensuring better alignment with biological neural activity.

**Implementation & Preprocessing**

* **Wrapper:** Used `hance-ai` wrapper in Hugging Face `transformers` to ensure strict adherence to original spectrogram protocols.
* **Input:** 2-second audio clips from video stimuli.
* **Spectrogram Specs:**
* Resample to  kHz.
* Mel-spectrogram: Window ms, Hop ms,  Mel bands.
* Patches:  grid (Time  Freq).



**Feature Extraction (Pooling)**

* **Goal:** Align granular model output with the averaged BOLD response (2s trial duration).
* **Extraction Point:** Final Transformer Block.
* **Tensor Shape:**  (Channels, Freq patches, Time patches).
* **Strategy:** **Global Average Pooling** across both  and  dimensions.
* **Final Output:** A single **768-dimensional vector** per stimulus, used to construct the Model RDM.

**Key References**

> * Huang, P. Y., et al. (2022). "Masked Autoencoders that Listen." *NeurIPS*.
> * Gemmeke, J. F., et al. (2017). "Audio Set." *ICASSP*.
> 
>

In [ ]:
from transformers import AutoModel
import torch
import torchaudio

# Path to the corrected download
AUDIOMAE_PATH = "/home/amin/hf_models/manual/AudioMAE"

# Load Model once (reuse for all segment durations)
model_mae = AutoModel.from_pretrained(
    AUDIOMAE_PATH, 
    local_files_only=False,
    trust_remote_code=True
).to(DTYPE).to(DEVICE).eval()

print(f"AudioMAE (Base) Ready")

# Loop through all segment durations
for seg_sec in SEGMENT_DURATIONS:
    print(f"\n{'='*60}")
    print(f"AudioMAE: Processing {int(seg_sec)}s segments")
    print(f"{'='*60}")
    
    # Gather data for this segment duration
    paired, all_videos, all_audios = gather_pairs_for_segment(seg_sec)
    
    if len(paired) == 0:
        print(f"No data found for {int(seg_sec)}s segments. Skipping.")
        continue
    
    print(f"Found {len(paired)} paired segments")
    
    # Create output directory
    out_dir = OUTPUT_PATH / "audiomae" / f"{int(seg_sec)}s"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    audiomae_embs = []
    print("Starting AudioMAE Extraction...")
    
    with torch.inference_mode():
        for i, ap in enumerate(all_audios):
            # Load audio with target length aligned to segment duration
            target_len = int(seg_sec * 16000)
            wav = load_audio(ap, target_len)
            
            # Check if model has an encoder attribute
            try:
                if hasattr(model_mae, 'encoder'):
                    # Get encoder output directly
                    outputs = model_mae.encoder.encode(ap, DEVICE)
                else:
                    outputs = model_mae(ap)
            except Exception:
                # Fallback: pass wav tensor
                outputs = model_mae(wav.unsqueeze(0).to(DEVICE))
            
            if i == 0:
                print(f"Output shape: {outputs.shape}")
            
            # Pool over ALL dimensions to get single vector
            while outputs.ndim > 1:
                outputs = outputs.mean(dim=1)  # Pool over freq and time
            
            embedding = outputs.unsqueeze(0)  # [768] -> [1, 768]
            audiomae_embs.append(embedding.cpu())
            
            if (i + 1) % 50 == 0:
                print(f"Processed {i + 1}/{len(all_audios)} audio segments...")

    audiomae_out = torch.cat(audiomae_embs, dim=0).numpy()
    print(f"AudioMAE embeddings shape: {audiomae_out.shape}")
    
    out_file = out_dir / "audiomae_a.npy"
    np.save(out_file, audiomae_out)
    print(f"Saved to: {out_file}\n")
    
    del audiomae_embs, audiomae_out
    gc.collect()

print("AudioMAE processing complete!")
del model_mae
gc.collect()
torch.cuda.empty_cache()


## PE-Core-Large Extraction

In [ ]:
import sys
REPO_ROOT = "/home/amin/hf_models/perception_models" 
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)
import torch
import torch.nn as nn
from torchvision.transforms.functional import to_pil_image
import core.vision_encoder.pe as pe
import core.vision_encoder.transforms as transforms
import os
import numpy as np

LOCAL_WEIGHTS_PATH = os.path.expanduser("/home/amin/.cache/huggingface/hub/models-facebook--PE-Core-L14/PE-Core-L14-336.pt")

# Load model once (reuse for all segment durations)
try:
    model = pe.CLIP.from_config("PE-Core-L14-336", pretrained=False)
    state_dict = torch.load(LOCAL_WEIGHTS_PATH, map_location='cpu')
    if "model" in state_dict:
        state_dict = state_dict["model"]
    model.load_state_dict(state_dict, strict=True)
    print(f"Weights loaded successfully.")
    model = model.to(DEVICE, dtype=DTYPE).eval()
except Exception as e:
    print(f"\nError: {e}")
    model = pe.CLIP.from_config("PE-Core-L14-336", pretrained=True).to(DEVICE).eval()

resolution = int(model.image_size)
preprocess = transforms.get_image_transform(resolution)
print(f"Using transform with resolution: {resolution}")

# Loop through all segment durations
for seg_sec in SEGMENT_DURATIONS:
    print(f"\n{'='*60}")
    print(f"PE-Core: Processing {int(seg_sec)}s segments")
    print(f"{'='*60}")
    
    # Gather data for this segment duration
    paired, all_videos, all_audios = gather_pairs_for_segment(seg_sec)
    
    if len(paired) == 0:
        print(f"No data found for {int(seg_sec)}s segments. Skipping.")
        continue
    
    print(f"Found {len(paired)} paired segments")
    
    # Create output directory
    out_dir = OUTPUT_PATH / "pe-core-l14" / f"{int(seg_sec)}s"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    pecore_embs = []
    model.eval()

    print(f"Starting extraction for {len(all_videos)} segments...")

    with torch.inference_mode():
        for i, vp in enumerate(all_videos):
            # 1. Load 16 frames: returns [16, 3, H, W]
            frames = load_video(vp) 
            
            # 2. Preprocess: Convert tensor frames to PIL, apply transforms, stack, and move to GPU
            # PE-Core requires 336x336 normalization
            processed_frames = torch.stack([
                preprocess(to_pil_image(f)) for f in frames
            ]).to(DEVICE, dtype=DTYPE)
            
            # 3. Vision Tower Forward Pass
            # Returns global features for each frame [16, 1024]
            visual_features = model.visual(processed_frames) 
            
            # 4. Temporal Pooling
            # Mean pool the 16 frames to get one 1024-dim vector for the 2s segment
            h = visual_features.mean(dim=0, keepdim=True) # [1, 1024]
            
            pecore_embs.append(h.cpu())
            
            if (i + 1) % 50 == 0:
                print(f"Processed {i + 1}/{len(all_videos)} segments...")

    # 5. Concatenate and Save
    pecore_out = torch.cat(pecore_embs).numpy()
    print(f"PE-Core-Large embeddings shape: {pecore_out.shape}")
    
    out_file = out_dir / "pe-core-l14_v.npy"
    np.save(out_file, pecore_out)
    print(f"Saved to: {out_file}\n")
    
    del pecore_embs, pecore_out
    gc.collect()

print("PE-Core processing complete!")
del model, preprocess
gc.collect()
torch.cuda.empty_cache()


## CLAP Extraction

In [ ]:
from transformers import ClapModel, ClapProcessor,AutoModel
import os

CLAP_PATH = "/home/amin/hf_models/manual/models--laion--larger_clap_music_and_speech/snapshots/195c3a3e68faebb3e2088b9a79e79b43ddbda76b"

# Load model once (reuse for all segment durations)

try:
    model = AutoModel.from_pretrained(
    CLAP_PATH, 
    local_files_only=True, 
    use_safetensors=True
).to(DTYPE).to(DEVICE).eval()

    state_dict = torch.load(LOCAL_WEIGHTS_PATH, map_location='cpu')
    if "model" in state_dict:
        state_dict = state_dict["model"]
    model.load_state_dict(state_dict, strict=True)
    print(f"Weights loaded successfully.")
    model = model.to(DEVICE, dtype=DTYPE).eval()
except Exception as e:
    print(f"\nError: {e}")
    model = ClapModel.from_pretrained(
    CLAP_PATH, 
    local_files_only=True, 
    use_safetensors=True
).to(DTYPE).to(DEVICE).eval()

print("CLAP Model & Processor Ready (Loaded from .bin)")

# Loop through all segment durations
for seg_sec in SEGMENT_DURATIONS:
    print(f"\n{'='*60}")
    print(f"CLAP: Processing {int(seg_sec)}s segments")
    print(f"{'='*60}")
    
    # Gather data for this segment duration
    paired, all_videos, all_audios = gather_pairs_for_segment(seg_sec)
    
    if len(paired) == 0:
        print(f"No data found for {int(seg_sec)}s segments. Skipping.")
        continue
    
    print(f"Found {len(paired)} paired segments")
    
    # Create output directory
    out_dir = OUTPUT_PATH / "clap-large" / f"{int(seg_sec)}s"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    clap_embs = []
    target_len = int(seg_sec * 48000)  # CLAP uses 48kHz

    with torch.inference_mode():
        for i, ap in enumerate(all_audios):
            wav = load_audio(ap, target_len, sr=48000)
            inputs = processor_clap(audios=wav.numpy(), sampling_rate=48000, return_tensors="pt")
            
            # Move tensors to device and dtype individually
            inputs = {k: v.to(DTYPE).to(DEVICE) if isinstance(v, torch.Tensor) else v 
                      for k, v in inputs.items()}
            
            out = model_clap.get_audio_features(**inputs)
            # CLAP audio features are already pooled, shape: (batch, hidden_size)
            clap_embs.append(out.cpu())
            
            if (i + 1) % 50 == 0:
                print(f"Processed {i + 1}/{len(all_audios)} audio segments...")

    clap_out = torch.cat(clap_embs).numpy()
    print(f"CLAP embeddings shape: {clap_out.shape}")
    
    out_file = out_dir / "clap-large_a.npy"
    np.save(out_file, clap_out)
    print(f"Saved to: {out_file}\n")
    
    del clap_embs, clap_out
    gc.collect()

print("CLAP processing complete!")
del model_clap, processor_clap
gc.collect()
torch.cuda.empty_cache()


## Whisper Extraction

In [ ]:
from transformers import WhisperModel, WhisperProcessor
import os

WHISPER_PATH = "/home/amin/hf_models/manual/models--openai--whisper-large-v3/snapshots/06f233fe06e710322aca913c1bc4249a0d71fce1"
LOCAL_FILES_ONLY = True

# Load model once (reuse for all segment durations)
try:
    if LOCAL_FILES_ONLY:
        model_whisper = WhisperModel.from_pretrained(
            WHISPER_PATH, 
            local_files_only=True
        ).to(DTYPE).to(DEVICE).eval()
        
        processor_whisper = WhisperProcessor.from_pretrained(
            WHISPER_PATH, 
            local_files_only=True
        )
    print("Whisper Model & Processor Ready (Sharded Safetensors Loaded)")
except Exception as e:
    print(f"Error: {e}")

# Loop through all segment durations
for seg_sec in SEGMENT_DURATIONS:
    print(f"\n{'='*60}")
    print(f"Whisper: Processing {int(seg_sec)}s segments")
    print(f"{'='*60}")
    
    # Gather data for this segment duration
    paired, all_videos, all_audios = gather_pairs_for_segment(seg_sec)
    
    if len(paired) == 0:
        print(f"No data found for {int(seg_sec)}s segments. Skipping.")
        continue
    
    print(f"Found {len(paired)} paired segments")
    
    # Create output directory
    out_dir = OUTPUT_PATH / "whisper-large-v3" / f"{int(seg_sec)}s"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    whisper_embs = []
    target_len = int(seg_sec * 16000)

    with torch.inference_mode():
        for i, ap in enumerate(all_audios):
            wav = load_audio(ap, target_len)
            inputs = processor_whisper(wav.numpy(), sampling_rate=16000, return_tensors="pt").to(DTYPE).to(DEVICE)
            # Get encoder outputs
            encoder_outputs = model_whisper.encoder(
                inputs.input_features,
                output_hidden_states=False,
                return_dict=True
            )
            # Mean pool over temporal dimension
            h = encoder_outputs.last_hidden_state.mean(dim=1)
            whisper_embs.append(h.cpu())
            if (i + 1) % 50 == 0:
                print(f"Processed {i + 1}/{len(all_audios)} audio segments...")

    whisper_out = torch.cat(whisper_embs).numpy()
    print(f"Whisper embeddings shape: {whisper_out.shape}")
    
    out_file = out_dir / "whisper-large-v3_a.npy"
    np.save(out_file, whisper_out)
    print(f"Saved to: {out_file}\n")
    
    del whisper_embs, whisper_out
    gc.collect()

print("Whisper processing complete!")
del model_whisper, processor_whisper
gc.collect()
torch.cuda.empty_cache()


## Next steps:
VideoMAE-base videomae-small, and other sizes of PE-Core (small and base) and wav-lm ( small and base) and whisper (small and base) should be downloaded when available; along with different sizes of PE-Core (large and giant have already been downloaded) to make sure the RSA results are consistent. I also have to study the backbone and structure of the alignment and the cores of the models to make sure that the similarities are drawn between similar models in terms of structure and size


| **Size Tier** | **Cornerstone (PE-AV)** | **A-V Control (Speech)** | **A-V Control (General, no text)** | **Visual Control (Reconstructive)** | **Visual Control (Text-Aligned)** | **Audio Control (Reconstructive)** | **Audio Control (Text-Aligned)** | **Audio Speech Control (Structure)** |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Small**<br>*(Audio: Base)* | **PE-AV-Small**<br>*(Audio: ViT-B, ~90M)*<br>*(Video: ViT-L, ~300M)* | **AV-HuBERT-Base**<br>*(ViT-Base, ~95M)* | **CAV-MAE-Sync**<br>*(ViT-Base, ~86M)*<br>*(2025 SOTA)* | **VideoMAE-v2-Large**<br>*(ViT-L, ~300M)*<br>*(Matches PE-L)* | **PE-Core-Large**<br>*(PE-Core-L14-336)*<br>*(~300M)* | **AudioMAE-Base**<br>*(ViT-B, ~86M)*<br>*(Perfect Match)* | **LAION-CLAP**<br>*(HTSAT-Base)*<br>*(~19M)* | **WavLM-Base+**<br>*(ViT-Base, ~95M)*<br>*(Structure Match)* |
| **Base**<br>*(Audio: Large)* | **PE-AV-Base**<br>*(Audio: ViT-L, ~210M)*<br>*(Video: ViT-L, ~300M)* | **AV-HuBERT-Large**<br>*(ViT-Large, ~316M)*<br>*(Good Match)* | *CAV-MAE-Sync*<br>*(Base: Under-match)* | **VideoMAE-v2-Large**<br>*(ViT-L, ~300M)*<br>*(Perfect Match)* | **PE-Core-Large**<br>*(PE-Core-L14-336)*<br>*(~300M)* | *AudioMAE-Base*<br>*(Mismatch: ~86M)* | **LAION-CLAP**<br>*(HTSAT-Large)*<br>*(or Large-Fuse)* | **WavLM-Large**<br>*(ViT-Large, ~317M)*<br>*(Structure Match)* |
| **Large**<br>*(Audio: Giant)* | **PE-AV-Large**<br>*(Audio: ViT-Giant, ~1.1B)*<br>*(Video: ViT-L, ~300M)* | *AV-HuBERT-Large*<br>*(Under-match)* | **ImageBind-Huge**<br>*(ViT-H, ~630M)*<br>*(Closest A-V Match)* | **VideoMAE-v2-Large**<br>*(ViT-L, ~300M)*<br>*(Perfect Match)* | **PE-Core-Large**<br>*(PE-Core-L14-336)*<br>*(~300M)* | *AudioMAE-Base*<br>*(Major Mismatch)* | *CLAP-Large*<br>*(Under-match)* | **Whisper-Large-v3**<br>*(ViT-Giant, ~1.55B)*<br>*(Size Match)* |
| **Giant**<br>*(Visual: Giant)* | **N/A**<br>*(Baseline Reference)* | *None* | **ImageBind-Huge**<br>*(ViT-H, ~630M)* | **VideoMAE-v2-Giant**<br>*(ViT-g, ~1B)* | **PE-Core-Giant**<br>*(PE-Core-G14)*<br>*(~1.9B)* | *None* | *None* | **Whisper-Large-v3**<br>*(~1.55B)* |